<a href="https://colab.research.google.com/github/Guliko24/PubMed_Fetch/blob/main/Week1_Day3_vector_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

# 1. Delete any existing corrupted files
for file in ["search_utils.py", "search_utils.html"]:
    if os.path.exists(file):
        os.remove(file)
        print(f"🗑️ Deleted old/corrupted {file}")

# 2. The clean, pure Python code for our utility file
clean_code = '''import json
import os
import numpy as np
from typing import List, Dict, Any
from Bio import Entrez
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

Entrez.email = "sesimboyle@gmail.com"
print("⏳ Loading Sentence Transformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Model loaded.\\n")

def load_or_fetch_pubmed_data(
    filename: str = "pubmed_abstracts.json",
    query: str = '(\\'single-cell\\' OR \\'spatial transcriptomics\\') AND \\'human brain\\'',
    max_records: int = 10
) -> List[Dict[str, Any]]:
    documents = []
    if os.path.exists(filename):
        try:
            with open(filename, "r", encoding="utf-8") as f:
                documents = json.load(f)
            if len(documents) > 0:
                print(f"✅ Loaded {len(documents)} documents from '{filename}'.")
                return documents
        except Exception as e:
            print(f"⚠️ Error reading file: {e}. Will re-fetch.")

    print(f"🔄 Fetching data from PubMed...")
    search_handle = Entrez.esearch(db="pubmed", term=query, retmax=max_records, sort="relevance")
    search_results = Entrez.read(search_handle)
    search_handle.close()
    pmids = search_results.get("IdList", [])

    if not pmids:
        print("❌ No PMIDs found.")
        return []

    fetch_handle = Entrez.efetch(db="pubmed", id=pmids, rettype="abstract", retmode="xml")
    records = Entrez.read(fetch_handle)
    fetch_handle.close()

    extracted_data = []
    articles_list = records.get('PubmedArticleSet', {}).get('PubmedArticle', [])
    if isinstance(articles_list, dict):
        articles_list = [articles_list]

    for article in articles_list:
        try:
            pmid_node = article["MedlineCitation"]["PMID"]
            pmid = pmid_node.get("text", "") if isinstance(pmid_node, dict) else str(pmid_node)

            title_raw = article["MedlineCitation"]["Article"]["ArticleTitle"]
            title = title_raw.get("content", "No Title") if isinstance(title_raw, dict) else title_raw

            abstract_node = article["MedlineCitation"]["Article"].get("Abstract", {})
            abstract_text_raw = abstract_node.get("AbstractText", [])

            if isinstance(abstract_text_raw, str):
                abstract_text = abstract_text_raw
            elif isinstance(abstract_text_raw, list):
                abstract_parts = [part.get('content', '') if isinstance(part, dict) else str(part) for part in abstract_text_raw]
                abstract_text = " ".join(abstract_parts).strip()
            else:
                abstract_text = "No abstract available"

            extracted_data.append({"pmid": pmid, "title": title, "abstract": abstract_text})
        except KeyError as e:
            print(f"⚠️ Skipping record due to missing key: {e}")
            continue

    if extracted_data:
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(extracted_data, f, indent=2)
        print(f"✅ Successfully fetched and saved {len(extracted_data)} documents!")
    return extracted_data

def build_bm25_index(documents: List[Dict[str, Any]]) -> BM25Okapi:
    tokenized_corpus = [doc["abstract"].lower().split() for doc in documents]
    return BM25Okapi(tokenized_corpus)

def search_bm25(bm25_index: BM25Okapi, documents: List[Dict[str, Any]], query: str, top_k: int = 5) -> List[Dict[str, Any]]:
    tokenized_query = query.lower().split()
    scores = bm25_index.get_scores(tokenized_query)
    ranked_results = sorted(zip(scores, documents), key=lambda x: x[0], reverse=True)
    return [{"score": float(score), "doc": doc} for score, doc in ranked_results[:top_k]]

def generate_embeddings(texts: List[str]) -> np.ndarray:
    print(f"🧠 Generating embeddings for {len(texts)} texts...")
    return model.encode(texts, show_progress_bar=False, normalize_embeddings=True)

def search_vectors(query: str, doc_embeddings: np.ndarray, documents: List[Dict[str, Any]], top_k: int = 5) -> List[Dict[str, Any]]:
    query_embedding = model.encode([query], normalize_embeddings=True)
    similarities = np.dot(doc_embeddings, query_embedding.T).flatten()
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return [{"score": float(similarities[idx]), "doc": documents[idx]} for idx in top_indices]
'''

# 3. Write the clean code to the file
with open("search_utils.py", "w", encoding="utf-8") as f:
    f.write(clean_code)

print("✅ search_utils.py has been cleanly written to the Colab directory!")

✅ search_utils.py has been cleanly written to the Colab directory!


In [2]:
# 1. Install dependencies
!pip install biopython rank-bm25 sentence-transformers numpy --quiet

# 2. Import the fresh file
from search_utils import (
    load_or_fetch_pubmed_data,
    build_bm25_index,
    search_bm25,
    generate_embeddings,
    search_vectors
)

# 3. Execute
docs = load_or_fetch_pubmed_data()

if docs:
    print("\n" + "="*60)
    print("🔍 TESTING BM25 (Lexical Search)")
    print("="*60)
    bm25 = build_bm25_index(docs)
    for r in search_bm25(bm25, docs, "microglia alzheimer disease", top_k=3):
        print(f"Score: {r['score']:.4f} | {r['doc']['title'][:60]}...")

    print("\n" + "="*60)
    print("🧠 TESTING VECTOR (Semantic Search)")
    print("="*60)
    embeddings = generate_embeddings([d["abstract"] for d in docs])
    for r in search_vectors("neurodegenerative disease mechanisms", embeddings, docs, top_k=3):
        print(f"Score: {r['score']:.4f} | {r['doc']['title'][:60]}...")
else:
    print("❌ No documents loaded. Check the output above for errors.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.3 MB/s eta 0:00:00
⏳ Loading Sentence Transformer model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model loaded.

🔄 Fetching data from PubMed...
❌ No documents loaded. Check the output above for errors.


In [1]:
!pip install biopython
import json
from Bio import Entrez

# 1. NCBI requires an email address to track usage
Entrez.email = "sesimboyle@gmail.com" # <-- REPLACE THIS WITH YOUR REAL EMAIL

# 2. Define the search query (tailored to our target roles)
query = "(single-cell OR spatial transcriptomics) AND (human brain) AND (novel cell type)"
max_records = 10 # Starting with 10 to test. We can increase to 50 later.

print(f"Searching PubMed for: {query}\n")

# 3. Search for PubMed IDs (PMIDs)
search_handle = Entrez.esearch(db="pubmed", term=query, retmax=max_records, sort="relevance")
search_results = Entrez.read(search_handle)
search_handle.close()

pmids = search_results["IdList"]
print(f"Found {len(pmids)} records. PMIDs: {pmids}\n")

# 4. Fetch the actual abstracts for these PMIDs
if pmids:
    fetch_handle = Entrez.efetch(db="pubmed", id=pmids, rettype="abstract", retmode="xml")
    records = Entrez.read(fetch_handle)
    fetch_handle.close()

    # 5. Extract and structure the data we care about
    extracted_data = []

    # Entrez.read for efetch with retmode='xml' typically returns a dictionary
    # where the top-level key is 'PubmedArticleSet', containing a list or dictionary of 'PubmedArticle's.
    articles_list = []
    if 'PubmedArticleSet' in records and 'PubmedArticle' in records['PubmedArticleSet']:
        articles_list = records['PubmedArticleSet']['PubmedArticle']
        if isinstance(articles_list, dict): # Handle case where only one article is returned as a dict
            articles_list = [articles_list]
    elif 'PubmedArticle' in records: # Sometimes Entrez.read might directly return this if PubmedArticleSet is implicit
        articles_list = records['PubmedArticle']
        if isinstance(articles_list, dict): # Handle single article case
            articles_list = [articles_list]

    for article in articles_list:
        # Accessing keys with correct capitalization as returned by Entrez.read from XML
        pmid_node = article["MedlineCitation"]["PMID"]
        if isinstance(pmid_node, dict):
            pmid = pmid_node.get("text", "")
        else: # it's likely a string
            pmid = str(pmid_node)

        # Safely get the article title
        title_raw = article["MedlineCitation"]["Article"]["ArticleTitle"]
        if isinstance(title_raw, dict): # Sometimes NCBI returns a dict with language tags
            title = title_raw.get("content", "No Title")
        else:
            title = title_raw

        # Safely get the abstract text
        abstract_node = article["MedlineCitation"]["Article"].get("Abstract", {})
        abstract_text_raw = abstract_node.get("AbstractText", [])

        if isinstance(abstract_text_raw, str):
            abstract_text = abstract_text_raw
        elif isinstance(abstract_text_raw, list):
            # AbstractText can be a list of dicts (e.g., {'label': '...', 'content': '...'}) or strings
            abstract_parts = []
            for part in abstract_text_raw:
                if isinstance(part, dict):
                    abstract_parts.append(part.get('content', ''))
                else:
                    abstract_parts.append(str(part))
            abstract_text = " ".join(abstract_parts).strip()
        else:
            abstract_text = ""

        extracted_data.append({
            "pmid": pmid,
            "title": title,
            "abstract": abstract_text
        })

    # 6. Save to a JSON file in the Colab environment
    output_filename = "pubmed_abstracts.json"
    with open(output_filename, "w", encoding="utf-8") as f:
        json.dump(extracted_data, f, indent=2)

    print(f"✅ Success! Saved {len(extracted_data)} abstracts to '{output_filename}'")

    # 7. Preview the first abstract to verify it worked
    if extracted_data: # Only preview if data exists
        print("\n--- PREVIEW OF FIRST ABSTRACT ---")
        print(f"PMID: {extracted_data[0]['pmid']}")
        print(f"Title: {extracted_data[0]['title']}")
        print(f"Abstract: {extracted_data[0]['abstract'][:300]}...") # Prints first 300 characters
    else:
        print("No abstracts were extracted.")
else:
    print("❌ No records found. Try adjusting the search query.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.8 MB/s eta 0:00:00
Searching PubMed for: (single-cell OR spatial transcriptomics) AND (human brain) AND (novel cell type)

Found 10 records. PMIDs: ['34582785', '40585969', '36544231', '40701154', '38350725', '39217332', '41456076', '41053013', '10940631', '11847215']

✅ Success! Saved 10 abstracts to 'pubmed_abstracts.json'

--- PREVIEW OF FIRST ABSTRACT ---
PMID: 34582785
Title: Single-nucleus transcriptome analysis reveals cell-type-specific molecular signatures across reward circuitry in the human brain.
Abstract: Single-cell gene expression technologies are powerful tools to study cell types in the human brain, but efforts have largely focused on cortical brain regions. We therefore created a single-nucleus RNA-sequencing resource of 70,615 high-quality nuclei to generate a molecular taxonomy of cell types a...


In [6]:
import os
import sys

# --- BEGIN FIX FOR search_utils.py CORRUPTION ---
# The clean, pure Python code for our utility file
clean_code = '''import json
import os
import numpy as np
from typing import List, Dict, Any
from Bio import Entrez
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

Entrez.email = "sesimboyle@gmail.com"
print("⏳ Loading Sentence Transformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Model loaded.")

def load_or_fetch_pubmed_data(
    filename: str = "pubmed_abstracts.json",
    query: str = '(\'single-cell\' OR \'spatial transcriptomics\') AND \'human brain\'',
    max_records: int = 10
) -> List[Dict[str, Any]]:
    documents = []
    if os.path.exists(filename):
        try:
            with open(filename, "r", encoding="utf-8") as f:
                documents = json.load(f)
            if len(documents) > 0:
                print(f"✅ Loaded {len(documents)} documents from '{filename}'.")
                return documents
        except Exception as e:
            print(f"⚠️ Error reading file: {e}. Will re-fetch.")

    print(f"🔄 Fetching data from PubMed...")
    search_handle = Entrez.esearch(db="pubmed", term=query, retmax=max_records, sort="relevance")
    search_results = Entrez.read(search_handle)
    search_handle.close()
    pmids = search_results.get("IdList", [])

    if not pmids:
        print("❌ No PMIDs found.")
        return []

    fetch_handle = Entrez.efetch(db="pubmed", id=pmids, rettype="abstract", retmode="xml")
    records = Entrez.read(fetch_handle)
    fetch_handle.close()

    extracted_data = []
    articles_list = records.get('PubmedArticleSet', {}).get('PubmedArticle', [])
    if isinstance(articles_list, dict):
        articles_list = [articles_list]

    for article in articles_list:
        try:
            pmid_node = article["MedlineCitation"]["PMID"]
            pmid = pmid_node.get("text", "") if isinstance(pmid_node, dict) else str(pmid_node)

            title_raw = article["MedlineCitation"]["Article"]["ArticleTitle"]
            title = title_raw.get("content", "No Title") if isinstance(title_raw, dict) else title_raw

            abstract_node = article["MedlineCitation"]["Article"].get("Abstract", {})
            abstract_text_raw = abstract_node.get("AbstractText", [])

            if isinstance(abstract_text_raw, str):
                abstract_text = abstract_text_raw
            elif isinstance(abstract_text_raw, list):
                abstract_parts = [part.get('content', '') if isinstance(part, dict) else str(part) for part in abstract_text_raw]
                abstract_text = " ".join(abstract_parts).strip()
            else:
                abstract_text = "No abstract available"

            extracted_data.append({"pmid": pmid, "title": title, "abstract": abstract_text})
        except KeyError as e:
            print(f"⚠️ Skipping record due to missing key: {e}")
            continue

    if extracted_data:
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(extracted_data, f, indent=2)
        print(f"✅ Successfully fetched and saved {len(extracted_data)} documents!")
    return extracted_data

def build_bm25_index(documents: List[Dict[str, Any]]) -> BM25Okapi:
    tokenized_corpus = [doc["abstract"].lower().split() for doc in documents]
    return BM25Okapi(tokenized_corpus)

def search_bm25(bm25_index: BM25Okapi, documents: List[Dict[str, Any]], query: str, top_k: int = 5) -> List[Dict[str, Any]]:
    tokenized_query = query.lower().split()
    scores = bm25_index.get_scores(tokenized_query)
    ranked_results = sorted(zip(scores, documents), key=lambda x: x[0], reverse=True)
    return [{"score": float(score), "doc": doc} for score, doc in ranked_results[:top_k]]

def generate_embeddings(texts: List[str]) -> np.ndarray:
    print(f"🧠 Generating embeddings for {len(texts)} texts...")
    return model.encode(texts, show_progress_bar=False, normalize_embeddings=True)

def search_vectors(query: str, doc_embeddings: np.ndarray, documents: List[Dict[str, Any]], top_k: int = 5) -> List[Dict[str, Any]]:
    query_embedding = model.encode([query], normalize_embeddings=True)
    similarities = np.dot(doc_embeddings, query_embedding.T).flatten()
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return [{"score": float(similarities[idx]), "doc": documents[idx]} for idx in top_indices]
'''

# 3. Write the clean code to the file
with open("search_utils.py", "w", encoding="utf-8") as f:
    f.write(clean_code)

print("✅ search_utils.py has been cleanly written to the Colab directory by current cell!")
# --- END FIX FOR search_utils.py CORRUPTION ---

# 1. Install dependencies
!pip install biopython rank-bm25 sentence-transformers numpy --quiet

# Ensure the current directory is in sys.path for module imports
if './' not in sys.path:
    sys.path.insert(0, './')

# 2. Import the functions
from search_utils import (
    load_or_fetch_pubmed_data,
    build_bm25_index,
    search_bm25,
    generate_embeddings,
    search_vectors
)

# 3. Execute
docs = load_or_fetch_pubmed_data()

if docs:
    print("\n" + "="*60)
    print("TESTING BM25 (Lexical Search)")
    print("="*60)
    bm25 = build_bm25_index(docs)
    for r in search_bm25(bm25, docs, "microglia alzheimer disease", top_k=3):
        print(f"Score: {r['score']:.4f} | {r['doc']['title'][:60]}...")

    print("\n" + "="*60)
    print("TESTING VECTOR (Semantic Search)")
    print("="*60)
    embeddings = generate_embeddings([d["abstract"] for d in docs])
    for r in search_vectors("neurodegenerative disease mechanisms", embeddings, docs, top_k=3):
        print(f"Score: {r['score']:.4f} | {r['doc']['title'][:60]}...")
else:
    print("No documents loaded.")

✅ search_utils.py has been cleanly written to the Colab directory by current cell!


SyntaxError: unterminated string literal (detected at line 12) (search_utils.py, line 12)